28x28x1 -> 5 channels conv 3x3, P = 0, S = 1 <br>
26x26x1 -> 10 channels conv 3x3, P = 0, S = 1 <br>
24x24x1 -> max pool 2x2, P = 0, S = 2 <br>
12x12x1 -> 12 channels conv 3x3, P = 0, S = 1 <br>
10x10x1 -> 15 channels conv 3x3, P = 0, S = 1 <br>
8x8x1 max pool 2x2, P = 0, S = 2 <br>
4x4x1 -> fully connected 40 nodes -> fully connected 10 nodes (for 10 labels) <br>

In [11]:
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using device: {torch.cuda.get_device_name(device)}")
else:
    device = "cpu"
    print(f"Using device: CPU")

Using device: Tesla T4


In [12]:
import torch
import torch.nn.functional as F
class moderu(torch.nn.Module):
    def __init__(self):
        super(moderu,self).__init__()
        self.conv1 = torch.nn.Conv2d(1,5,3) #input: 1 28x28 channel, kernel 3x3 -> output: 5 features for 5 26x26 channels
        self.conv2 = torch.nn.Conv2d(5,10,3) #input: 5 26x26 channels, kernel 3x3 -> output: 10 features for 10 24x24 channels (maxpooling 2x2 -> 10 12x12 channels)
        self.conv3 = torch.nn.Conv2d(10,12,3) #input: 10 12x12 channels, kernel 3x3 -> output: 15 features for 15 10x10 channels
        self.conv4 = torch.nn.Conv2d(12,15,3) #input: 15 10x10 channels, kernel 3x3 -> output: 5 features for 5 8x8 channels (maxpooling 2x2 -> 5 4x4 channels)
        self.lin1 = torch.nn.Linear(15*4*4,40) #fully connected layer, 40 nodes
        self.lin2 = torch.nn.Linear(40,10) #fully connected layer, 10 nodes (for 10 labels)
    def forward(self,x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(F.relu(self.conv2(x)),2)
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(F.relu(self.conv4(x)),2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.lin1(x))
        x = F.softmax(self.lin2(x), dim = 1)
        return x
    def num_flat_features(self,x):
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

In [13]:
!pip3 install torchinfo
import torchinfo
from torchinfo import summary
model = moderu().to(device)
print(model)
print(summary(model))

moderu(
  (conv1): Conv2d(1, 5, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(5, 10, kernel_size=(3, 3), stride=(1, 1))
  (conv3): Conv2d(10, 12, kernel_size=(3, 3), stride=(1, 1))
  (conv4): Conv2d(12, 15, kernel_size=(3, 3), stride=(1, 1))
  (lin1): Linear(in_features=240, out_features=40, bias=True)
  (lin2): Linear(in_features=40, out_features=10, bias=True)
)
Layer (type:depth-idx)                   Param #
moderu                                   --
├─Conv2d: 1-1                            50
├─Conv2d: 1-2                            460
├─Conv2d: 1-3                            1,092
├─Conv2d: 1-4                            1,635
├─Linear: 1-5                            9,640
├─Linear: 1-6                            410
Total params: 13,287
Trainable params: 13,287
Non-trainable params: 0


load model

load dataset

In [14]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm import tqdm
#transform
transform = transforms.Compose([
    transforms.ToTensor(),               # Convert images to PyTorch tensors
    transforms.Normalize((0.5,), (0.5,))  # Normalize the images with mean=0.5 and std=0.5
])

# load nmist dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform) #array of tensors
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform) #array of tensors

# Split training dataset into training and validation datasets
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

# dataloader for batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [15]:
# from tqdm import tqdm

train model

In [16]:
# criterion = torch.nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
# num_epochs = 10


# for epoch in range(num_epochs):

#     model.train()
#     for inputs, labels in tqdm(train_loader):
#         inputs, labels = inputs.to(device), labels.to(device)
#         optimizer.zero_grad() #zero the param gradient
#         outputs = model(inputs) #feed forward
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
#         #running_loss += loss.item()
#         #_, predicted = torch.max(outputs.data, 1)
#         # total += labels.size(0)
#         # correct += (predicted == labels).sum().item()
#     model.eval()
#     with torch.no_grad():
#       correct = 0
#       total = 0
#       running_loss = 0.0
#       for inputs, labels in tqdm(test_loader):
#           inputs, labels = inputs.to(device), labels.to(device)
#           outputs = model(inputs) #feed forward
#           loss = criterion(outputs, labels)
#           running_loss += loss.item()
#           _, predicted = torch.max(outputs.data, 1)
#           total += labels.size(0)
#           correct += (predicted == labels).sum().item()
#       # Print loss and accuracy statistics after each epoch
#       print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(test_loader):.4f}, Accuracy: {100 * correct / total:.2f}%")


# Model, Loss, Optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
num_epochs = 10

for epoch in range(num_epochs):
    # Training
    model.train()
    correct_train = 0
    total_train = 0
    running_loss_train = 0.0
    for inputs, labels in tqdm(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss_train += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    # Validation
    model.eval()
    correct_val = 0
    total_val = 0
    running_loss_val = 0.0
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_val += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    # Test
    correct_test = 0
    total_test = 0
    running_loss_test = 0.0
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_test += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    # Print loss and accuracy for each dataset
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Training   - Loss: {running_loss_train / len(train_loader):.4f}, Accuracy: {100 * correct_train / total_train:.2f}%")
    print(f"Validation - Loss: {running_loss_val / len(val_loader):.4f}, Accuracy: {100 * correct_val / total_val:.2f}%")
    print(f"Test       - Loss: {running_loss_test / len(test_loader):.4f}, Accuracy: {100 * correct_test / total_test:.2f}%")


100%|██████████| 313/313 [00:02<00:00, 123.64it/s]


Epoch [1/10]
Training   - Loss: 2.3023, Accuracy: 10.30%
Validation - Loss: 2.3023, Accuracy: 12.40%
Test       - Loss: 2.3022, Accuracy: 12.53%


100%|██████████| 313/313 [00:02<00:00, 126.03it/s]


Epoch [2/10]
Training   - Loss: 2.3021, Accuracy: 10.93%
Validation - Loss: 2.3021, Accuracy: 10.00%
Test       - Loss: 2.3020, Accuracy: 10.33%


100%|██████████| 313/313 [00:03<00:00, 101.11it/s]


...

Epoch [8/10]
Training   - Loss: 1.6680, Accuracy: 79.50%
Validation - Loss: 1.6350, Accuracy: 83.02%
Test       - Loss: 1.6376, Accuracy: 82.73%


100%|██████████| 313/313 [00:03<00:00, 94.67it/s] 


Epoch [9/10]
Training   - Loss: 1.6381, Accuracy: 82.50%
Validation - Loss: 1.6772, Accuracy: 78.30%
Test       - Loss: 1.6710, Accuracy: 78.93%


100%|██████████| 313/313 [00:02<00:00, 120.46it/s]

Epoch [10/10]
Training   - Loss: 1.6214, Accuracy: 84.09%
Validation - Loss: 1.6032, Accuracy: 85.87%
Test       - Loss: 1.6048, Accuracy: 85.78%


In [17]:
from google.colab import files
torch.save(model.state_dict(), 'model.pth')
# files.download('model.pth')

In [18]:
trained_moderu = torch.load('model.pth')
# print(trained_moderu)

<ipython-input-18-ac4944f155be>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  trained_moderu = torch.load('model.pth')


In [19]:
import sys
import numpy as np
np.set_printoptions(threshold=sys.maxsize)
# trained_moderu['conv1.weight'].cpu().numpy().tofile('conv1_weight.txt', sep=', ', format='%s')
# trained_moderu['conv1.bias'].cpu().numpy().tofile('conv1_bias.txt', sep=', ', format='%s')
# trained_moderu['conv2.weight'].cpu().numpy().tofile('conv2_weight.txt', sep=', ', format='%s')
# trained_moderu['conv2.bias'].cpu().numpy().tofile('conv2_bias.txt', sep=', ', format='%s')
# trained_moderu['conv3.weight'].cpu().numpy().tofile('conv3_weight.txt', sep=', ', format='%s')
# trained_moderu['conv3.bias'].cpu().numpy().tofile('conv3_bias.txt', sep=', ', format='%s')
# trained_moderu['conv4.weight'].cpu().numpy().tofile('conv4_weight.txt', sep=', ', format='%s')
# trained_moderu['conv4.bias'].cpu().numpy().tofile('conv4_bias.txt', sep=', ', format='%s')
# trained_moderu['lin1.weight'].cpu().numpy().tofile('lin1_weight.txt', sep=', ', format='%s')
# trained_moderu['lin1.bias'].cpu().numpy().tofile('lin1_bias.txt', sep=', ', format='%s')
# trained_moderu['lin2.weight'].cpu().numpy().tofile('lin2_weight.txt', sep=', ', format='%s')
# trained_moderu['lin2.bias'].cpu().numpy().tofile('lin2_bias.txt', sep=', ', format='%s')
# f = open("result.txt", "w")
# f.write("Data")
# f.close()
# f = open("result.txt", "a")
# f.write(trained_moderu['conv1.bias'].cpu().numpy())
# files.download('conv1_weight.txt')
# files.download('conv1_bias.txt')
# files.download('conv2_weight.txt')
# files.download('conv2_bias.txt')
# files.download('conv3_weight.txt')
# files.download('conv3_bias.txt')
# files.download('conv4_weight.txt')
# files.download('conv4_bias.txt')
# files.download('lin1_weight.txt')
# files.download('lin1_bias.txt')
# files.download('lin2_weight.txt')
# files.download('lin2_bias.txt')
